# L71 · Whisper 解码策略 — 贪婪解码与 beam search 从原理到实现

**学习目标**
- 理解自回归（autoregressive，AR）解码：每步用之前的 token 预测下一个
- 实现贪婪解码（greedy）和简单 beam search（width=2）
- 了解 Whisper 的特殊 token 体系

## 自回归解码：token-by-token 生成

 Whisper 输出的是 token 概率序列。解码 = 从概率分布中反复采样，直到遇到结束符。

**贪婪解码**：每步选概率最大的 token。快，O(T)，但容易陷入重复。

**Beam search（宽度 W）**：保留 W 条最优路径，每步扩展 W×vocab_size 个候选，保留前 W 个。慢（O(T×W×vocab_size)），质量更好。

### Whisper 的特殊 token
```
<|startoftranscript|>  — 解码起始符
<|en|>                 — 语言标签（英语）
<|transcribe|>         — 任务标签
<|notimestamps|>       — 不输出时间戳
<|endoftext|>          — 解码结束符
```
解码时强制在序列开头插入这些特殊 token，引导模型行为。

In [ ]:
import numpy as np

In [ ]:
# 模拟一个玩具 LM：词汇表大小 10，5 步解码
np.random.seed(7)
VOCAB_SIZE = 10
EOT = 9            # end-of-text token id
MAX_STEPS = 8

def fake_lm(context: list) -> np.ndarray:
    """玩具语言模型：基于上文输出 logits。（模拟，真实中是 Transformer 前向传播）"""
    rng = np.random.RandomState(sum(context) % 137)
    logits = rng.randn(VOCAB_SIZE)
    if len(context) >= 4:
        logits[EOT] += 2.0   # 超过 4 步后更容易结束
    return logits

def softmax(logits):
    x = logits - logits.max()
    e = np.exp(x)
    return e / e.sum()

## ✏️ 任务 1：贪婪解码

In [ ]:
def greedy_decode(prompt: list, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    """贪婪解码：每步选概率最大的 token。"""
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

prompt = [0, 1, 2]   # 模拟 [<|startoftranscript|>, <|en|>, <|transcribe|>]
result = greedy_decode(prompt)
print(f'贪婪解码结果: {result}')
assert result[0:3] == prompt, '前缀应保持不变'
print('✅ 贪婪解码验证通过')

## ✏️ 任务 2：Beam Search（宽度 W=2）

In [ ]:
def beam_search(prompt: list, width: int = 2, max_steps: int = MAX_STEPS, eot: int = EOT) -> list:
    """简单 beam search。返回分数最高的序列。"""
    # ✏️ TODO: 实现此函数
    raise NotImplementedError("TODO")

beam_result = beam_search(prompt, width=2)
print(f'Beam search 结果 (W=2): {beam_result}')
print(f'贪婪结果:               {result}')
print('（两者可能不同——beam search 探索更多路径）')

## 本课收束

| 策略 | 复杂度 | 特点 |
|---|---|---|
| 贪婪 | O(T×V) | 快，局部最优 |
| Beam W=2 | O(T×W×V) | 更好，代价是 W 倍计算 |
| Top-k / Top-p | O(T×V) | 引入随机性，避免重复（见 L86）|

Whisper 默认使用 beam_size=5 + length_penalty，生产环境中效果最好。

下一步 L72：在真实数据集上用 LoRA 微调 Whisper。